In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)

1.Basics: estimator Api, datasets, train_test_split

In [2]:

from sklearn.datasets import load_iris

iris = load_iris()
x, y = iris.data, iris.target

print("X shape:", x.shape)     # (150 samples, 4 features)
print("y shape:", y.shape)
print("Features:", iris.feature_names)
print("Classes:", list(iris.target_names))

df = pd.DataFrame(x, columns=iris.feature_names)
df["species"] = pd.Series(y).map(dict(enumerate(iris.target_names)))
df.head()

X shape: (150, 4)
y shape: (150,)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


Train/ test split

In [3]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", x_train.shape, "Test:", x_test.shape)
print("Class counts in train:", np.bincount(y_train))
print("Class counts in test:", np.bincount(y_test))

Train: (120, 4) Test: (30, 4)
Class counts in train: [40 40 40]
Class counts in test: [10 10 10]


THe 'fit->predict->score' pattern

In [4]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)   #1. create the model with settings(hyperparameters)
model.fit(x_train, y_train)                   #2. learn from training
preds = model.predict(x_test)                 #3. predict on unseen data

print("First 10 predictions:", preds[:10])
print("First 10 real labels:", y_test[:10])
print("Accuracy:", model.score(x_test, y_test))

First 10 predictions: [0 2 1 1 0 1 0 0 2 1]
First 10 real labels: [0 2 1 1 0 1 0 0 2 1]
Accuracy: 1.0


The 'transform' side of API 

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)      # learn mean/std from TRAIN and apply
x_test_s  = scaler.transform(x_test)           # only apply, never fit on test

print("Train mean after scaling:", x_train_s.mean(axis=0))         # ~0
print("Train std after  scaling:", x_train_s.std(axis=0))          # ~1
print("Learned means :", scaler.mean_)

Train mean after scaling: [-0. -0.  0.  0.]
Train std after  scaling: [1. 1. 1. 1.]
Learned means : [5.842 3.048 3.77  1.205]


2.Preprocessing: scaling, encoding, missing values, feature selection

In [7]:
data = pd.DataFrame({
    "age":    [25, 32, np.nan, 45, 29, 51, np.nan, 38],
    "salary": [30000, 52000, 41000, np.nan, 38000, 90000, 45000, 61000],
    "city":   ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Delhi", "Pune", "Mumbai"],
    "bought": ["no", "yes", "no", "yes", "no", "yes", "no", "yes"],
})

print(data)
print("\nMissing values per column:\n", data.isna().sum())

    age   salary    city bought
0  25.0  30000.0    Pune     no
1  32.0  52000.0  Mumbai    yes
2   NaN  41000.0    Pune     no
3  45.0      NaN   Delhi    yes
4  29.0  38000.0  Mumbai     no
5  51.0  90000.0   Delhi    yes
6   NaN  45000.0    Pune     no
7  38.0  61000.0  Mumbai    yes

Missing values per column:
 age       2
salary    1
city      0
bought    0
dtype: int64


2.1 Missing values: 'SimpleImputer'

In [11]:
from sklearn.impute import SimpleImputer

num_cols = ["age", "salary"]
imputer = SimpleImputer(strategy="median")
data[num_cols] = imputer.fit_transform(data[num_cols])
print(data)
print("\nstill missing:", data.isna().sum().sum())

    age   salary    city bought
0  25.0  30000.0    Pune     no
1  32.0  52000.0  Mumbai    yes
2  35.0  41000.0    Pune     no
3  45.0  45000.0   Delhi    yes
4  29.0  38000.0  Mumbai     no
5  51.0  90000.0   Delhi    yes
6  35.0  45000.0    Pune     no
7  38.0  61000.0  Mumbai    yes

still missing: 0


2.2 Scaling: 'StandardDcalar vs MinMaxScalar'

In [12]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

std_scaled = StandardScaler().fit_transform(data[num_cols])
mm_scaled = MinMaxScaler().fit_transform(data[num_cols])

compare = pd.DataFrame({
    "age (raw)": data["age"], "age (standard)": std_scaled[:, 0], "age (minmax)": mm_scaled[:, 0],
})
compare

,age (raw),age (standard),age (minmax)
0,25.0,-1.427,0.000
1,32.0,-0.539,0.269
2,35.0,-0.159,0.385
3,45.0,1.110,0.769
4,29.0,-0.919,0.154
5,51.0,1.870,1.000
6,35.0,-0.159,0.385
7,38.0,0.222,0.500


2.3 Encoding categories

'OneHotEncoder' and 'LabelEncoder'

In [14]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# one-hot for the feature 'city'
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")   #ignore = unseen city wont crash
city_encoded = ohe.fit_transform(data[["city"]])
city_df = pd.DataFrame(city_encoded, columns=ohe.get_feature_names_out(["city"]))
print(city_df.head())

# Label encoding for the target
le = LabelEncoder()
y_enc = le.fit_transform(data["bought"])
print("\nClasses:", le.classes_, "->", y_enc)
print("Back to text:", le.inverse_transform(y_enc[:3]))

   city_Delhi  city_Mumbai  city_Pune
0         0.0          0.0        1.0
1         0.0          1.0        0.0
2         0.0          0.0        1.0
3         1.0          0.0        0.0
4         0.0          1.0        0.0

Classes: ['no' 'yes'] -> [0 1 0 1 0 1 0 1]
Back to text: ['no' 'yes' 'no']


2.4 Feature selection

In [15]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
xc, yc = cancer.data, cancer.target
print("Original features:", xc.shape[1])

#a) remove near-constant feature
vt = VarianceThreshold(threshold=0.01)
print("After VarianceThreshold:", vt.fit_transform(xc).shape[1])

#b) keep the 5 best features by ANOVA F-score
skb = SelectKBest(score_func=f_classif, k=5)
xc_best = skb.fit_transform(xc, yc)
best_names = np.array(cancer.feature_names)[skb.get_support()]
print("Top 5 features:", list(best_names))

Original features: 30
After VarianceThreshold: 14
Top 5 features: [np.str_('mean perimeter'), np.str_('mean concave points'), np.str_('worst radius'), np.str_('worst perimeter'), np.str_('worst concave points')]


3. Regression(predicting a number)




In [23]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

diabetes = load_diabetes()

xd = diabetes.data
yd = diabetes.target

xd_tr, xd_te, yd_tr, yd_te = train_test_split(xd, yd, test_size=0.2, random_state =42)

#diabetes feature are already scaled. on your data, scale first for Ridge/Lasso.